# Day 2: Mini Metadata Validation (Teaching Demo)

This notebook implements a **lightweight** metadata completeness check.

**Important:** This is a teaching demo, not a formal validator.

## What we check
- `title` exists
- `creator` exists
- `identifier` exists
- `description` exists
- at least one of `publisher`, `publicationYear`, or `date` exists

### Colab tip
If the relative repository paths are not available, upload the three metadata files and use the `/content/...` fallback paths used in the first code cell.


In [ ]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd

# ---- Notebook-local parsers + validator (Colab/Jupyter friendly) ----
def _local_name(tag):
    return tag.split('}', 1)[1] if '}' in tag else tag

def parse_dc(path):
    root = ET.parse(path).getroot()
    def first(name):
        for el in root.iter():
            if _local_name(el.tag) == name:
                txt = (el.text or '').strip()
                if txt:
                    return txt
        return None
    creators = []
    for el in root.iter():
        if _local_name(el.tag) == 'creator':
            txt = (el.text or '').strip()
            if txt:
                creators.append(txt)
    return {
        'standard': 'Dublin Core',
        'title': first('title'),
        'creator': creators,
        'identifier': first('identifier'),
        'description': first('description'),
        'publisher': first('publisher'),
        'date': first('date'),
    }

def parse_datacite(path):
    root = ET.parse(path).getroot()
    out = {'standard': 'DataCite', 'creator': []}
    for el in root.iter():
        name = _local_name(el.tag)
        txt = (el.text or '').strip()
        if name == 'identifier' and txt and 'identifier' not in out:
            out['identifier'] = txt
        elif name == 'creatorName' and txt:
            out['creator'].append(txt)
        elif name == 'title' and txt and 'title' not in out:
            out['title'] = txt
        elif name == 'description' and txt and 'description' not in out:
            out['description'] = txt
        elif name == 'publisher' and txt and 'publisher' not in out:
            out['publisher'] = txt
        elif name == 'publicationYear' and txt and 'publicationYear' not in out:
            out['publicationYear'] = txt
    out.setdefault('title', None)
    out.setdefault('identifier', None)
    out.setdefault('description', None)
    out.setdefault('publisher', None)
    out.setdefault('publicationYear', None)
    return out

def parse_schema(path):
    obj = json.loads(Path(path).read_text(encoding='utf-8'))
    creators = []
    for c in obj.get('creator', []):
        if isinstance(c, dict) and c.get('name'):
            creators.append(c['name'])
    return {
        'standard': 'schema.org',
        'title': obj.get('name'),
        'creator': creators,
        'identifier': obj.get('identifier'),
        'description': obj.get('description'),
        'publisher': obj.get('publisher', {}).get('name') if isinstance(obj.get('publisher'), dict) else obj.get('publisher'),
    }

def _has_nonempty(value):
    if value is None:
        return False
    if isinstance(value, list):
        return any(str(v).strip() for v in value)
    return bool(str(value).strip())

def completeness_score(meta):
    checks = [
        ('title', _has_nonempty(meta.get('title'))),
        ('creator', _has_nonempty(meta.get('creator'))),
        ('identifier', _has_nonempty(meta.get('identifier'))),
        ('description', _has_nonempty(meta.get('description'))),
        ('publisher_or_year', _has_nonempty(meta.get('publisher')) or _has_nonempty(meta.get('publicationYear')) or _has_nonempty(meta.get('date'))),
    ]
    passed = sum(1 for _, ok in checks if ok)
    return {
        'standard': meta.get('standard'),
        'score_percent': round((passed / len(checks)) * 100, 1),
        'checks': checks,
    }

# Local repo paths with Colab upload fallback
dc_path = Path('../../data/metadata/climate_dataset_dublin_core.xml')
dcite_path = Path('../../data/metadata/climate_dataset_datacite.xml')
schema_path = Path('../../data/metadata/climate_dataset_schemaorg.jsonld')
if not dc_path.exists():
    dc_path = Path('/content/climate_dataset_dublin_core.xml')
if not dcite_path.exists():
    dcite_path = Path('/content/climate_dataset_datacite.xml')
if not schema_path.exists():
    schema_path = Path('/content/climate_dataset_schemaorg.jsonld')

dc_meta = parse_dc(dc_path)
datacite_meta = parse_datacite(dcite_path)
schema_meta = parse_schema(schema_path)

scores = [completeness_score(dc_meta), completeness_score(datacite_meta), completeness_score(schema_meta)]
scores

In [ ]:
score_rows = [
    {"standard": s["standard"], "completeness_score_percent": s["score_percent"]}
    for s in scores
]

score_df = pd.DataFrame(score_rows).sort_values("completeness_score_percent", ascending=False)
score_df

## Interpretation (teaching perspective)

In practice, completeness checks help identify missing fields early.

To improve Reusability, students typically focus on:
- richer `description` and clear variable meaning,
- clear `publisher` and consistent `identifier` usage,
- and any documentation that helps others interpret the dataset correctly.
